In [ ]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 5.1 MB/s eta 0:00:00


In [ ]:
# Scenario
# “A hospital uses different employees (agents) to handle patient care in sequence.”

# ================================
# AGENT 1: Intake Agent (Planner)
# - Collects patient symptoms and history
# - Decides which steps are needed (tests, consultations, etc.)
# ================================

# ================================
# AGENT 2: Diagnostic Agent
# - Orders lab tests or scans
# - Interprets results and identifies possible conditions
# ================================

# ================================
# AGENT 3: Treatment Agent
# - Suggests treatment options (medication, therapy, surgery)
# - Considers patient preferences and medical guidelines
# ================================

# ================================
# AGENT 4: Decision Agent
# - Reviews all inputs (history, diagnostics, treatment options)
# - Provides the final recommendation to the patient
# ================================

import json
from typing import Dict, Any
from groq import Groq


GROQ_API_KEY = "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"
GROQ_MODEL = "llama-3.3-70b-versatile"

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found")

client = Groq(api_key=GROQ_API_KEY)


def call_llm(system_prompt: str, user_prompt: str) -> str:
    completion = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.2,
        max_completion_tokens=1200
    )
    return completion.choices[0].message.content.strip()


def safe_json_parse(text: str) -> Dict[str, Any]:
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        start = text.find("{")
        end = text.rfind("}") + 1
        if start != -1 and end != -1 and end > start:
            return json.loads(text[start:end])
        raise ValueError(f"Model did not return valid JSON. Raw response: {text}")


class IntakeAgent:
    def run(self, patient_data: Dict[str, Any]) -> Dict[str, Any]:
        print("\n[Agent 1: Intake Agent] Reviewing patient history and planning next steps...")

        system_prompt = (
            "You are Agent 1 in a hospital sequential multi-agent system. "
            "Your job is intake planning only. "
            "Read the patient case, summarize the situation, identify the next steps, "
            "set urgency, and keep the answer practical and structured. "
            "Return only valid JSON with exactly these keys: "
            "summary, required_steps, priority_level, intake_notes. "
            "Keep required_steps short and realistic. "
            "Do not include any markdown, explanation outside JSON, or extra keys."
        )

        user_prompt = f"""
Patient Case:
{json.dumps(patient_data, indent=2)}

Output rules:
- summary: 2 to 3 lines
- required_steps: short list of immediate steps
- priority_level: low, medium, or high
- intake_notes: concise operational notes
"""

        result = call_llm(system_prompt, user_prompt)
        return safe_json_parse(result)


class DiagnosticAgent:
    def simulate_tests(self, patient_data: Dict[str, Any], intake_output: Dict[str, Any]) -> Dict[str, Any]:
        symptoms = [s.lower() for s in patient_data.get("symptoms", [])]
        vitals = patient_data.get("vitals", {})
        temperature = vitals.get("temperature_c", 98.6)
        spo2 = vitals.get("spo2", 99)
        heart_rate = vitals.get("heart_rate", 80)

        lab_results = {
            "cbc": {
                "wbc": "high" if "fever" in symptoms or temperature > 99.5 else "normal",
                "hemoglobin": "normal"
            },
            "crp": "elevated" if "fever" in symptoms else "normal",
            "blood_sugar": "normal"
        }

        scan_results = {
            "chest_xray": "mild infiltrates" if "cough" in symptoms and spo2 < 97 else "clear",
            "ecg": "mild sinus tachycardia" if heart_rate > 95 else "normal"
        }

        return {
            "ordered_tests": intake_output.get("required_steps", []),
            "lab_results": lab_results,
            "scan_results": scan_results
        }

    def run(self, patient_data: Dict[str, Any], intake_output: Dict[str, Any]) -> Dict[str, Any]:
        print("\n[Agent 2: Diagnostic Agent] Ordering tests and identifying possible conditions...")

        test_data = self.simulate_tests(patient_data, intake_output)

        system_prompt = (
            "You are Agent 2 in a hospital sequential multi-agent system. "
            "Your role is diagnostic interpretation only. "
            "Use the patient details and simulated tests to identify likely conditions. "
            "Return only valid JSON with exactly these keys: "
            "possible_conditions, diagnostic_reasoning, confidence_level, red_flags. "
            "Keep the response realistic, professional, and concise. "
            "Do not include markdown or extra keys."
        )

        user_prompt = f"""
Patient Data:
{json.dumps(patient_data, indent=2)}

Intake Output:
{json.dumps(intake_output, indent=2)}

Simulated Test Data:
{json.dumps(test_data, indent=2)}

Output rules:
- possible_conditions: list of 2 to 4 likely conditions
- diagnostic_reasoning: 4 to 6 lines
- confidence_level: low, medium, medium-high, or high
- red_flags: short list
"""

        result = call_llm(system_prompt, user_prompt)
        diagnosis = safe_json_parse(result)
        diagnosis["test_data"] = test_data
        return diagnosis


class TreatmentAgent:
    def run(self, patient_data: Dict[str, Any], diagnosis_output: Dict[str, Any]) -> Dict[str, Any]:
        print("\n[Agent 3: Treatment Agent] Preparing treatment options based on diagnosis and preferences...")

        system_prompt = (
            "You are Agent 3 in a hospital sequential multi-agent system. "
            "Your role is treatment planning only. "
            "Use diagnosis, patient preferences, allergy information, and general safety principles. "
            "Return only valid JSON with exactly these keys: "
            "treatment_options, preferred_option, treatment_rationale, follow_up_advice. "
            "Keep treatment options broad, practical, and suitable for a hospital simulation. "
            "Avoid unnecessary over-detail. "
            "Do not include markdown or extra keys."
        )

        user_prompt = f"""
Patient Data:
{json.dumps(patient_data, indent=2)}

Diagnosis Output:
{json.dumps(diagnosis_output, indent=2)}

Output rules:
- treatment_options: 2 to 4 options
- preferred_option: one selected option
- treatment_rationale: 4 to 6 lines
- follow_up_advice: concise and practical
- consider allergy and patient preference for home recovery if possible
"""

        result = call_llm(system_prompt, user_prompt)
        return safe_json_parse(result)


class DecisionAgent:
    def run(
        self,
        patient_data: Dict[str, Any],
        intake_output: Dict[str, Any],
        diagnosis_output: Dict[str, Any],
        treatment_output: Dict[str, Any]
    ) -> Dict[str, Any]:
        print("\n[Agent 4: Decision Agent] Reviewing all inputs and generating final recommendation...")

        system_prompt = (
            "You are Agent 4 in a hospital sequential multi-agent system. "
            "Your role is final decision making. "
            "Review the intake, diagnostics, and treatment suggestions and deliver the final recommendation. "
            "Return only valid JSON with exactly these keys: "
            "final_recommendation, patient_friendly_explanation, escalation_needed, final_notes. "
            "Keep the answer professional, readable, and operationally useful. "
            "Do not include markdown or extra keys."
        )

        user_prompt = f"""
Patient Data:
{json.dumps(patient_data, indent=2)}

Intake Output:
{json.dumps(intake_output, indent=2)}

Diagnosis Output:
{json.dumps(diagnosis_output, indent=2)}

Treatment Output:
{json.dumps(treatment_output, indent=2)}

Output rules:
- final_recommendation: 3 to 5 lines
- patient_friendly_explanation: simple language
- escalation_needed: true or false
- final_notes: concise internal note
"""

        result = call_llm(system_prompt, user_prompt)
        return safe_json_parse(result)


class HospitalMultiAgentSystem:
    def __init__(self):
        self.intake_agent = IntakeAgent()
        self.diagnostic_agent = DiagnosticAgent()
        self.treatment_agent = TreatmentAgent()
        self.decision_agent = DecisionAgent()

    def run(self, patient_data: Dict[str, Any]) -> Dict[str, Any]:
        print("Patient Case Received:", patient_data.get("name", "Unknown Patient"))

        intake_output = self.intake_agent.run(patient_data)
        diagnosis_output = self.diagnostic_agent.run(patient_data, intake_output)
        treatment_output = self.treatment_agent.run(patient_data, diagnosis_output)
        decision_output = self.decision_agent.run(
            patient_data,
            intake_output,
            diagnosis_output,
            treatment_output
        )

        return {
            "patient_data": patient_data,
            "intake_output": intake_output,
            "diagnosis_output": diagnosis_output,
            "treatment_output": treatment_output,
            "decision_output": decision_output
        }


patient_case = {
    "name": "Rahul Sharma",
    "age": 45,
    "gender": "Male",
    "symptoms": ["fever", "cough", "fatigue"],
    "medical_history": ["hypertension"],
    "allergies": ["penicillin"],
    "vitals": {
        "temperature_c": 101.2,
        "blood_pressure": "140/90",
        "heart_rate": 96,
        "spo2": 95
    },
    "preferences": {
        "prefers_non_surgical": True,
        "prefers_home_recovery_if_possible": True
    }
}

system = HospitalMultiAgentSystem()
output = system.run(patient_case)

print("\n" + "=" * 70)
print("HOSPITAL SEQUENTIAL MULTI-AGENT SYSTEM OUTPUT")
print("=" * 70)

print("\n1. Intake Agent Output")
print(json.dumps(output["intake_output"], indent=2))

print("\n2. Diagnostic Agent Output")
print(json.dumps(output["diagnosis_output"], indent=2))

print("\n3. Treatment Agent Output")
print(json.dumps(output["treatment_output"], indent=2))

print("\n4. Decision Agent Output")
print(json.dumps(output["decision_output"], indent=2))

print("\n" + "=" * 70)
print("FINAL SYSTEM SUMMARY")
print("=" * 70)
print("This is a sequential multi-agent hospital workflow simulation.")
print("Agent 1 handled intake planning.")
print("Agent 2 handled diagnostics and test interpretation.")
print("Agent 3 handled treatment planning.")
print("Agent 4 handled the final decision.")
print("Each agent used the previous agent's output, which makes this a pipeline-style multi-agent system.")

Patient Case Received: Rahul Sharma

[Agent 1: Intake Agent] Reviewing patient history and planning next steps...

[Agent 2: Diagnostic Agent] Ordering tests and identifying possible conditions...

[Agent 3: Treatment Agent] Preparing treatment options based on diagnosis and preferences...

[Agent 4: Decision Agent] Reviewing all inputs and generating final recommendation...

HOSPITAL SEQUENTIAL MULTI-AGENT SYSTEM OUTPUT

1. Intake Agent Output
{
  "summary": "Rahul Sharma, a 45-year-old male, presents with fever, cough, and fatigue. He has a history of hypertension and is allergic to penicillin. His current temperature is 101.2 degrees Celsius.",
  "required_steps": [
    "Take a complete medical history",
    "Order a chest X-ray",
    "Run a blood test"
  ],
  "priority_level": "medium",
  "intake_notes": "Monitor patient's oxygen saturation and blood pressure closely, consider isolation due to infectious symptoms"
}

2. Diagnostic Agent Output
{
  "possible_conditions": [
    "Pneu